In [1]:
import pandas as pd

df = pd.read_csv("../data/vscode_issues.csv")

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Shape: (1000, 7)

Columns:
['issue_id', 'title', 'body', 'labels', 'state', 'created_at', 'closed_at']


In [2]:
df.isnull().sum()

issue_id        0
title           0
body            7
labels        310
state           0
created_at      0
closed_at     517
dtype: int64

In [3]:
df["labels"].value_counts().head(30)

labels
new release                                                                      93
spam                                                                             91
info-needed                                                                      53
info-needed,new release                                                          27
chat-billing                                                                     20
testplan-item                                                                    16
agent-host                                                                       14
chat-billing,new release                                                         13
*duplicate                                                                       13
spam,new release                                                                 11
*duplicate,new release                                                           10
bug,error-telemetry,recent-regression,extension,ai-generated,ai-analy

In [4]:
for i in range(5):
    print("="*80)
    print("TITLE:", df.iloc[i]["title"])
    print("\nLABELS:", df.iloc[i]["labels"])
    print("\nBODY:")
    print(str(df.iloc[i]["body"])[:500])

TITLE: Unsaved image preview tab can't be closed (and is buggy)

LABELS: new release

BODY:

Type: <b>Bug</b>

1. Create a new unsaved file
2. Paste a valid SVG into the file (I used godotengine/godot/tree/master/editor/icons/Enum.svg)
3. Right-click the tab and select "Reopen Editor With..."
4. Select "Image Preview"
5. Ensure the file isn't saved
6. Try to close the tab (and select one of the saving options)

Note: It didn't matter which button I clicked. All three of "Save", "Don't save", and "Cancel" had the same result.

After creating this file, and clicking "Open file using VS C
TITLE: 已经升级GitHub Pro，依然受限制

LABELS: chat-billing,new release

BODY:

Type: <b>Bug</b>

已经完成升级，聊天配额还是受限制

VS Code version: Code 1.123.0 (Universal) (6a44c352bd24569c417e530095901b649960f9f8, 2026-06-03T11:29:03+02:00)
OS version: Darwin arm64 24.2.0
Modes:

<details>
<summary>System Info</summary>

|Item|Value|
|---|---|
|CPUs|Apple M4 (10 x 2400)|
|GPU Status|2d_canvas: enabled<br>GPU0: VENDOR= 0x106b [

In [5]:
df["body_length"] = df["body"].fillna("").str.len()

df["body_length"].describe()

count     1000.000000
mean      2619.723000
std       4003.206609
min          0.000000
25%        661.000000
50%       1495.500000
75%       3238.250000
max      53937.000000
Name: body_length, dtype: float64

In [6]:
import re
import pandas as pd

def clean_issue(text):
    if pd.isna(text):
        return ""

    # Remove HTML comments
    text = re.sub(r"<!--.*?-->", " ", text, flags=re.DOTALL)

    # Remove URLs
    text = re.sub(r"http\S+", " ", text)

    # Remove HTML tags
    text = re.sub(r"<.*?>", " ", text)

    # Remove markdown headings
    text = re.sub(r"#+", " ", text)

    # Remove repeated whitespace
    text = re.sub(r"\s+", " ", text)

    return text.strip()

In [7]:
df["clean_body"] = df["body"].apply(clean_issue)

In [8]:
df["document"] = (
    "Title: " +
    df["title"].fillna("") +
    "\n\nDescription: " +
    df["clean_body"].fillna("")
)

In [9]:
idx = 0

print("ORIGINAL\n")
print(df.loc[idx, "body"][:1000])

print("\n" + "="*80 + "\n")

print("CLEANED\n")
print(df.loc[idx, "document"][:1000])

ORIGINAL


Type: <b>Bug</b>

1. Create a new unsaved file
2. Paste a valid SVG into the file (I used godotengine/godot/tree/master/editor/icons/Enum.svg)
3. Right-click the tab and select "Reopen Editor With..."
4. Select "Image Preview"
5. Ensure the file isn't saved
6. Try to close the tab (and select one of the saving options)

Note: It didn't matter which button I clicked. All three of "Save", "Don't save", and "Cancel" had the same result.

After creating this file, and clicking "Open file using VS Code's standard text/binary editor?", the tab's icon was changed to the default language icon, and the tab's name was changed to Untitled-1. I created a new text tab, which was also named Untitled-1, and it appeared that the bugged image preview tab copied the title and icon of the newly created text tab, but only if the tab wasn't currently visible (i.e. with split screen, the title doesn't update until I focus a different tab on that screen).

Close All Editors doesn't work either.

V

In [10]:
duplicate_count = df["labels"].fillna("").str.contains(
    "duplicate",
    case=False
).sum()

print("Duplicate Issues:", duplicate_count)

Duplicate Issues: 24


In [11]:
df.to_csv(
    "../data/vscode_issues_clean.csv",
    index=False
)

In [12]:
!pip install sentence-transformers chromadb

  Using cached torch-2.12.0-cp312-cp312-win_amd64.whl.metadata (31 kB)
  Using cached scipy-1.17.1-cp312-cp312-win_amd64.whl.metadata (60 kB)
     ---------------------------------------- 0.0/109.4 kB ? eta -:--:--
     -------------------------------------- 109.4/109.4 kB 3.2 MB/s eta 0:00:00
     ---------------------------------------- 0.0/52.0 kB ? eta -:--:--
     ---------------------------------------- 52.0/52.0 kB 2.6 MB/s eta 0:00:00
  Using cached overrides-7.7.0-py3-none-any.whl.metadata (5.8 kB)
  Using cached bcrypt-5.0.0-cp39-abi3-win_amd64.whl.metadata (10 kB)
  Using cached tenacity-9.1.4-py3-none-any.whl.metadata (1.2 kB)
     ---------------------------------------- 0.0/43.1 kB ? eta -:--:--
     ---------------------------------------- 43.1/43.1 kB 2.2 MB/s eta 0:00:00
  Using cached rich-15.0.0-py3-none-any.whl.metadata (18 kB)
  Using cached filelock-3.29.1-py3-none-any.whl.metadata (2.0 kB)
  Using cached fsspec-2026.4.0-py3-none-any.whl.metadata (10 kB)
  Using c


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [14]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\keert\OneDrive\Desktop\issue-triage-assistant\venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\keert\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [15]:
embedding = model.encode(
    "Login fails after password reset"
)

print(type(embedding))
print(len(embedding))

<class 'numpy.ndarray'>
384


In [16]:
sample_df = df.head(100).copy()

embeddings = model.encode(
    sample_df["document"].tolist(),
    show_progress_bar=True
)

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

In [17]:
print(embeddings.shape)

(100, 384)


In [18]:
import chromadb

client = chromadb.Client()

collection = client.create_collection(
    name="vscode_issues"
)

In [19]:
sample_df = df.head(100).copy()

In [20]:
collection.add(
    documents=sample_df["document"].tolist(),
    ids=[str(i) for i in sample_df.index]
)

C:\Users\keert\.cache\chroma\onnx_models\all-MiniLM-L6-v2\onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:19<00:00, 4.37MiB/s]


In [21]:
results = collection.query(
    query_texts=[
        "Image preview editor cannot be closed"
    ],
    n_results=5
)

In [22]:
for doc in results["documents"][0]:
    print("=" * 80)
    print(doc[:500])

Title: Unsaved image preview tab can't be closed (and is buggy)

Description: Type: Bug 1. Create a new unsaved file 2. Paste a valid SVG into the file (I used godotengine/godot/tree/master/editor/icons/Enum.svg) 3. Right-click the tab and select "Reopen Editor With..." 4. Select "Image Preview" 5. Ensure the file isn't saved 6. Try to close the tab (and select one of the saving options) Note: It didn't matter which button I clicked. All three of "Save", "Don't save", and "Cancel" had the same r
Title: Non-functional version 1.123

Description: Type: Bug I was running version 1.122 I clicked ‘Update’ in VSCode The whole editor stopped working properly. The Git tree with modified files isn't loading. The Codex Sidebar panel won't open The extensions for custom file icons work sometimes but not others. Most importantly: I keep getting a message about running bisect. I am unable to work normally in the editor following the latest update VS Code version: Code 1.123.0 (6a44c352bd24569c417e5

In [23]:
import chromadb
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [24]:
embeddings = model.encode(
    df["document"].tolist(),
    show_progress_bar=True
)

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

In [25]:
client = chromadb.Client()

collection = client.create_collection(
    name="vscode_rag"
)

In [26]:
collection.add(
    ids=[str(i) for i in range(len(df))],
    documents=df["document"].tolist(),
    embeddings=embeddings.tolist()
)

In [27]:
query = "Image preview editor cannot be closed"

query_embedding = model.encode(query)

In [28]:
results = collection.query(
    query_embeddings=[query_embedding.tolist()],
    n_results=5
)

In [29]:
for doc in results["documents"][0]:
    print("="*80)
    print(doc[:500])

Title: Unsaved image preview tab can't be closed (and is buggy)

Description: Type: Bug 1. Create a new unsaved file 2. Paste a valid SVG into the file (I used godotengine/godot/tree/master/editor/icons/Enum.svg) 3. Right-click the tab and select "Reopen Editor With..." 4. Select "Image Preview" 5. Ensure the file isn't saved 6. Try to close the tab (and select one of the saving options) Note: It didn't matter which button I clicked. All three of "Save", "Don't save", and "Cancel" had the same r
Title: Move "Open as preview" button so that it's in the same spot as "Reopen as source file"

Description: The "Open as Preview" button in markdown files and the "Reopen as source file" button in the previews are in different spots. Switching back and forth between source and preview requires mouse movements. It would be great if those two buttons stayed in the same spot. Switching the positions of the "Open as Preview" and "Open Preview to the Side" buttons would probably solve the issue. In 

In [36]:
def retrieve_similar_issues(query, n_results=5):
    query_embedding = model.encode(query)

    results = collection.query(
        query_embeddings=[query_embedding.tolist()],
        n_results=n_results
    )

    return results

In [37]:
collection.add(
    ids=[str(i) for i in range(len(df))],
    documents=df["document"].tolist(),
    embeddings=embeddings.tolist(),
    metadatas=[
        {
            "issue_id": int(row["issue_id"]),
            "title": row["title"],
            "labels": str(row["labels"])
        }
        for _, row in df.iterrows()
    ]
)

In [38]:
results = retrieve_similar_issues(
    "Unable to login after password reset"
)

In [39]:
for meta in results["metadatas"][0]:
    print(meta["title"])

TypeError: 'NoneType' object is not subscriptable

In [41]:
print(results.keys())

dict_keys(['ids', 'embeddings', 'documents', 'uris', 'included', 'data', 'metadatas', 'distances'])


In [42]:
print(results["documents"][0][0][:300])

Title: the authentication is not working

Description: Type: Feature Request the authentication process of connecting github to my email account is not working what should i do? VS Code version: Code 1.122.1 (8761a5560cfd65fdd19ce7e2bd18dab5c0a4d84e, 2026-05-29T09:25:37+02:00) OS version: Windows_NT


In [40]:
print(results)

{'ids': [['330', '286', '697', '341', '841']], 'embeddings': None, 'documents': [['Title: the authentication is not working\n\nDescription: Type: Feature Request the authentication process of connecting github to my email account is not working what should i do? VS Code version: Code 1.122.1 (8761a5560cfd65fdd19ce7e2bd18dab5c0a4d84e, 2026-05-29T09:25:37+02:00) OS version: Windows_NT x64 10.0.19045 Modes: System Info |Item|Value| |---|---| |CPUs|AMD Athlon Silver 3050e with Radeon Graphics (4 x 1397)| |GPU Status|2d_canvas: enabled GPU0: VENDOR= 0x1002, DEVICE=0x15d8 [AMD Radeon(TM) Vega 3 Graphics], DRIVER_VENDOR=AMD, DRIVER_VERSION=31.0.12044.24003 *ACTIVE* GPU1: VENDOR= 0x1414, DEVICE=0x008c [Microsoft Basic Render Driver], DRIVER_VERSION=10.0.19041.5794 Machine model name: Machine model version: direct_rendering_display_compositor: disabled_off_ok gpu_compositing: enabled multiple_raster_threads: enabled_on opengl: enabled_on rasterization: enabled raw_draw: disabled_off_ok skia_gra

In [35]:
print(collection.count())

1000


In [47]:
import google.generativeai as genai

genai.configure(api_key=GEMINI_API_KEY)

for m in genai.list_models():
    if "generateContent" in m.supported_generation_methods:
        print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.5-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.5-preview
models/gemini-robotics-er-1.6-preview
models/gemini-2.5-computer-use-pr

In [48]:
import google.generativeai as genai

genai.configure(api_key=GEMINI_API_KEY)

model_gemini = genai.GenerativeModel("gemini-2.5-flash")

response = model_gemini.generate_content(
    "Explain what a software bug report is in one sentence."
)

print(response.text)

A software bug report is a detailed description of an error or defect found in software, provided to facilitate its resolution.


In [45]:
!pip install google-generativeai


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [49]:
def build_context(results):
    context = ""

    for i, doc in enumerate(results["documents"][0]):
        context += f"\n\nIssue {i+1}:\n"
        context += doc[:1500]

    return context

In [52]:
def analyze_issue(title, description):

    issue_text = f"""
    Title: {title}

    Description:
    {description}
    """

    retrieved = retrieve_similar_issues(issue_text)

    context = build_context(retrieved)

    prompt = f"""
You are a software issue triage assistant.

Current Issue:
{issue_text}

Similar Historical Issues:
{context}

Return ONLY valid JSON in this format:

{{
  "category": "",
  "severity": "",
  "duplicate_probability": "",
  "summary": "",
  "resolution": ""
}}
"""

    response = model_gemini.generate_content(prompt)

    return response.text

In [53]:
result = analyze_issue(
    "Login fails after password reset",
    """
    After resetting password,
    users cannot login.
    System shows invalid credentials.
    """
)

print(result)

```json
{
  "category": "Authentication/Authorization",
  "severity": "Critical",
  "duplicate_probability": "Low",
  "summary": "Users are unable to log in after successfully resetting their password, consistently receiving an 'invalid credentials' error. This indicates a fundamental failure in the system's ability to recognize or validate newly set user credentials.",
  "resolution": "Investigate the password reset workflow end-to-end. Focus on verifying: 1) Correct storage (hashing and saving) of the new password in the database. 2) Proper retrieval and validation of the new password by the authentication service during login attempts. 3) Any caching mechanisms that might be holding stale credentials. 4) Synchronization issues if multiple authentication points or databases are involved. Review authentication service logs for specific error details."
}
```
